<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用JAX 和Flax 對CodeGemma 進行推論

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/codegemma/codegemma_flax_inference"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/docs/codegemma/codegemma_flax_inference.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemma/cookbook/blob/main/docs/codegemma/codegemma_flax_inference.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemma%2Fcookbook%2Fmain%2Fdocs%2Fcodegemma%2Fcodegemma_flax_inference.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemma/cookbook/blob/main/docs/codegemma/codegemma_flax_inference.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

我們提出了 CodeGemma，這是一個基於 Google DeepMind 的 Gemma 模型的開放程式碼模型集合（Gemma Team et al., 2024）。
CodeGemma 是一系列輕量級、最先進的開放模型，採用與創建 Gemini 模型相同的研究和技術而構建。
繼續 Gemma 預訓練模型，CodeGemma 模型進一步訓練超過 500 到 10000 億個 tokens 主要代碼，使用
與 Gemma 型號系列具有相同的架構。因此，CodeGemma 模型在完成和完成方面都實現了最先進的程式碼效能
和發電任務，同時保持強勁
大規模的理解和推論能力。
CodeGemma 有 3 種變體：
* 7B 程式碼預訓練模型
* 7B 指令調整的程式碼模型
* 一個 2B 模型，專門針對程式碼填充和開放式生成進行訓練。

本指南將引導您使用 CodeGemma 模型和 Flax 來完成程式碼完成任務。
**注意：** 此notebook 在Google Colab 中的 TPU v2 上執行，因為 T4 GPU 記憶體不足。

## 設定

### 1. 為CodeGemma 設定Kaggle 存取權限

要完成本教學，您首先需要按照 [Gemma 設定](https://ai.google.dev/gemma/docs/setup) 中的設定說明進行操作，其中向您展示如何執行以下操作：
* 在 [kaggle.com](https://www.kaggle.com/models/google/codegemma/) 上造訪CodeGemma。
* 選擇資源充足的Colabruntime（**T4 GPU 記憶體不足，請使用 TPU v2取代**）來執行CodeGemma模型。
* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

完成 Gemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。
### 2.設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。當 prompted 並選擇「授予存取權限？」時訊息，同意提供secret存取。

In [ ]:
import os
from google.colab import userdata # `userdata` is a Colab API.

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 3. 安裝 `gemma` library

免費Colab硬體加速目前*不足以*執行此notebook。如果您使用的是 [Colab Pay As You Go 或 Colab Pro](https://colab.research.google.com/signup)，請點選 **編輯** > **notebook 設定** > 選擇 **A100 GPU** > **儲存** 以啟用硬體加速。
接下來，您需要從 [`github.com/google-deepmind/gemma`](https://github.com/google-deepmind/gemma) 安裝 Google DeepMind `gemma` library。如果您收到有關「pip 的依賴解析器」的錯誤，通常可以忽略它。
**注意：** 透過安裝 `gemma`，您也將安裝 [`flax`](https://flax.readthedocs.io)、核心 [`jax`](https://jax.readthedocs.io](`orbax`](@@P00007@0) 與 [@P@P@003@P@P00007@00)。

In [ ]:
!pip install -q git+https://github.com/google-deepmind/gemma.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 1.1 MB/s eta 0:00:00


### 4. 導入庫

這個notebook使用[Gemma](https://github.com/google-deepmind/gemma)（使用[Flax](https://flax.readthedocs.io)來建構其神經網路層）和[SentencePiece](https://github.com/google/sentencepiece)來建構其神經網路層）和[SentencePiece](https://github.com/google/sentencepiece)（用於化）。

In [ ]:
import os
from gemma import params as params_lib
from gemma import sampler as sampler_lib
from gemma import transformer as transformer_lib
import sentencepiece as spm

## 載入 CodeGemma 模型

使用 [`kagglehub.model_download`](https://github.com/Kaggle/kagglehub/blob/bddefc718182282882b72f814d407d89e5d178c4/src/kagglehub/models.py#L12) 載入 CodeGemma 模型，模型採用三個參數：
- `handle`：來自Kaggle的模型句柄
- `path`：（可選字串）本地路徑
- `force_download`：（可選布林值）強制重新下載模型

**注意：** 請注意，`2b-pt` 型號的大小約為 3.66Gb。

In [ ]:
GEMMA_VARIANT = '2b-pt' # @param ['2b-pt', '7b-it', '7b-pt', '1.1-2b-pt', '1.1-7b-it'] {type:"string"}

In [ ]:
import kagglehub

GEMMA_PATH = kagglehub.model_download(f'google/codegemma/flax/{GEMMA_VARIANT}')

100%|██████████| 3.67G/3.67G [00:22<00:00, 173MB/s]
Extracting model files...


In [ ]:
print('GEMMA_PATH:', GEMMA_PATH)

GEMMA_PATH: /root/.cache/kagglehub/models/google/codegemma/flax/2b-pt/3


**注意：** 上面輸出的路徑是模型權重和tokenizer本地保存的位置，稍後您將需要它們。

檢查模型權重和tokenizer的位置，然後設定路徑變數。 tokenizer 目錄將位於您下載模型的主目錄中，而模型權重將位於子目錄中。例如：
- `spm.model` tokenizer 檔案將位於 `/LOCAL/PATH/TO/codegemma/flax/2b-pt/3` 中
- 模型 checkpoint 將位於`/LOCAL/PATH/TO/codegemma/flax/2b-pt/3/2b-pt`

In [ ]:
CKPT_PATH = os.path.join(GEMMA_PATH, GEMMA_VARIANT[-5:])
TOKENIZER_PATH = os.path.join(GEMMA_PATH, 'spm.model')
print('CKPT_PATH:', CKPT_PATH)
print('TOKENIZER_PATH:', TOKENIZER_PATH)

CKPT_PATH: /root/.cache/kagglehub/models/google/codegemma/flax/2b-pt/3/2b-pt
TOKENIZER_PATH: /root/.cache/kagglehub/models/google/codegemma/flax/2b-pt/3/spm.model


## 執行採樣/inference

使用 [`gemma.params.load_and_format_params`](https://github.com/google-deepmind/gemma/blob/c6bd156c246530e1620a7c62de98542a377e3934/gemma/params.py#L27) 方法載入並格式化 CodeGemma 模型checkpoint：

In [ ]:
params = params_lib.load_and_format_params(CKPT_PATH)

載入 CodeGemma tokenizer，使用 [`sentencepiece.SentencePieceProcessor`](https://github.com/google/sentencepiece/blob/4d6a1f41069c4636c51a5590f7578a0dbed83450/python/src/sentencepiece/__init__.py#L423) 建構：

In [ ]:
vocab = spm.SentencePieceProcessor()
vocab.Load(TOKENIZER_PATH)

True

若要從 CodeGemma 型號 checkpoint 自動載入正確的設定，請使用 [`gemma.transformer.TransformerConfig`](https://github.com/google-deepmind/gemma/blob/56e501ce147af4ea5c23cc0ddf5a9c4a6b7bd0d0/gemma/transformer.py#L65)。 `cache_size` 參數是 CodeGemma `Transformer` 快取中的時間步數。然後，使用 [`gemma.transformer.Transformer`](https://github.com/google-deepmind/gemma/blob/56e501ce147af4ea5c23cc0ddf5a9c4a6b7bd0d0/gemma/transformer.py#L136)（繼承自 [`flax.linen.Module`](https://flax.readthedocs.io/en/latest/api_reference/flax.linen/module.html)）將 CodeGemma 模型實例化為`model_2b`。
**注意：** 由於目前CodeGemma 版本中未使用tokens，因此詞彙表大小小於輸入嵌入的數量。

In [ ]:
transformer_config = transformer_lib.TransformerConfig.from_params(
    params,
    cache_size=1024
)

transformer = transformer_lib.Transformer(config=transformer_config)

使用 [`gemma.sampler.Sampler`](https://github.com/google-deepmind/gemma/blob/56e501ce147af4ea5c23cc0ddf5a9c4a6b7bd0d0/gemma/sampler.py#L88) 建立`sampler`。它使用CodeGemma 型號checkpoint 和tokenizer。

In [ ]:
sampler = sampler_lib.Sampler(
    transformer=transformer,
    vocab=vocab,
    params=params['transformer']
)

建立一些變數來表示中間填滿 (fim) tokens 並建立一些輔助函數來格式化 prompt 和產生的輸出。
例如，我們來看看下面的程式碼：```
def function(string):
assert function('asdf') == 'fdsa'
```
我們想要填寫`function`，以便斷言成立`True`。在這種情況下，前綴將是：```
"def function(string):\n"
```
後綴是：```
"assert function('asdf') == 'fdsa'"
```
然後我們將其格式化為 prompt 作為 PREFIX-SUFFIX-MIDDLE （需要填充的中間部分始終位於 prompt 的末尾）：```
"<|fim_prefix|>def function(string):\n<|fim_suffix|>assert function('asdf') == 'fdsa'<|fim_middle|>"
```

In [ ]:
# In the context of a code editor,
# the cursor is the location where the text will be inserted
BEFORE_CURSOR = "<|fim_prefix|>"
AFTER_CURSOR = "<|fim_suffix|>"
AT_CURSOR = "<|fim_middle|>"
FILE_SEPARATOR = "<|file_separator|>"

def format_completion_prompt(before, after):
  print(f"\nORIGINAL PROMPT:\n{before}{after}")
  prompt = f"{BEFORE_CURSOR}{before}{AFTER_CURSOR}{after}{AT_CURSOR}"
  print(f"\nFORMATTED PROMPT:\n{repr(prompt)}")
  return prompt
def format_generated_output(before, after, output):
  print(f"\nGENERATED OUTPUT:\n{repr(output)}")
  formatted_output = f"{before}{output.replace(FILE_SEPARATOR, '')}{after}"
  print(f"\nFILL-IN COMPLETION:\n{formatted_output}")
  return formatted_output

建立prompt並執行inference。指定前綴`before` 文本和後綴`after` 文本，並使用輔助函數`format_completion prompt` 產生格式化的prompt。
您可以調整`total_generation_steps`（產生回應時執行的步驟數 - 本範例使用`100`保留主機記憶體）。
**注意：**如果內存不足，請按一下**執行時** > **斷開連接並刪除runtime**，然後單擊**執行時** > **全部執行**。

In [ ]:
before = "def function(string):\n"
after = "assert function('asdf') == 'fdsa'"
prompt = format_completion_prompt(before, after)

output = sampler(
    [prompt],
    total_generation_steps=100,
    ).text

formatted_output = format_generated_output(before, after, output[0])


ORIGINAL PROMPT:
def function(string):
assert function('asdf') == 'fdsa'

FORMATTED PROMPT:
"<|fim_prefix|>def function(string):\n<|fim_suffix|>assert function('asdf') == 'fdsa'<|fim_middle|>"

GENERATED OUTPUT:
'    return string[::-1]\n\n<|file_separator|>'

FILL-IN COMPLETION:
def function(string):
    return string[::-1]

assert function('asdf') == 'fdsa'


In [ ]:
before = "import "
after = """if __name__ == "__main__":\n    sys.exit(0)"""
prompt = format_completion_prompt(before, after)

output = sampler(
    [prompt],
    total_generation_steps=100,
    ).text

formatted_output = format_generated_output(before, after, output[0])


ORIGINAL PROMPT:
import if __name__ == "__main__":
    sys.exit(0)

FORMATTED PROMPT:
'<|fim_prefix|>import <|fim_suffix|>if __name__ == "__main__":\n    sys.exit(0)<|fim_middle|>'

GENERATED OUTPUT:
'sys\n<|file_separator|>'

FILL-IN COMPLETION:
import sys
if __name__ == "__main__":
    sys.exit(0)


In [ ]:
before = """import numpy as np
def reflect(matrix):
  # horizontally reflect a matrix
"""
after = ""
prompt = format_completion_prompt(before, after)

output = sampler(
    [prompt],
    total_generation_steps=100,
    ).text

formatted_output = format_generated_output(before, after, output[0])


ORIGINAL PROMPT:
import numpy as np
def reflect(matrix):
  # horizontally reflect a matrix


FORMATTED PROMPT:
'<|fim_prefix|>import numpy as np\ndef reflect(matrix):\n  # horizontally reflect a matrix\n<|fim_suffix|><|fim_middle|>'

GENERATED OUTPUT:
'  return np.flip(matrix, axis=1)\n<|file_separator|>'

FILL-IN COMPLETION:
import numpy as np
def reflect(matrix):
  # horizontally reflect a matrix
  return np.flip(matrix, axis=1)



## 了解更多

- 您可以了解有關 Google DeepMind 的更多資訊 [`gemma` library on GitHub](https://github.com/google-deepmind/gemma)，其中包含您在本教學中使用的模組的文檔字串，例如 [`gemma.params`](https://github.com/google-deepmind/gemma/blob/main/gemma/params.py)
[`gemma.transformer`](https://github.com/google-deepmind/gemma/blob/main/gemma/transformer.py)，以及
[`gemma.sampler`](https://github.com/google-deepmind/gemma/blob/main/gemma/sampler.py)。- 以下庫有自己的文件網站：[core JAX](https://jax.readthedocs.io)、[Flax](https://flax.readthedocs.io) 和 [Orbax](https://orbax.readthedocs.io/)。
- 有關 `sentencepiece` tokenizer/detokenizer 文檔，請查看 [Google 的 `sentencepiece` GitHub 儲存庫](https://github.com/google/sentencepiece)。
- 對於 `kagglehub` 文檔，請查看 [Kaggle 的 `kagglehub` GitHub 存儲庫](https://github.com/Kaggle/kagglehub) 上的 `README.md`。
- 了解如何[將 Gemma 模型與 Google Cloud Vertex AI 一起使用](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma)。
- 如果您使用 Google Cloud TPU（v3-8 及更高版本），請確保同時更新至最新的 `jax[tpu]` 軟體包 (`!pip install -U jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html`)，重新啟動 runtime，並檢查 `jax` 和 @@P0003@P 版本是否符合 (4@P.004@P0003@P 版本。這可以防止由於`jaxlib` 和`jax` 版本不匹配而出現`RuntimeError`。有關JAX的更多安裝說明，請參閱[JAX文件](https://jax.readthedocs.io/en/latest/tutorials/installation.html#install-google-tpu)。